# Encrypted Math Tutorial: Number Theory (`math/number_theory.py`)

This tutorial covers `src/concrete_fhe_toolkit/math/number_theory.py`. Number theory operations (like GCD, LCM, or primality testing) are mathematically complex and require dynamic branching. To perform these in FHE, we evaluate them across the bounded domain during compilation and inject them as cryptographically secure lookup tables.

## 1. GCD, LCM, Coprime, and Divisibility
These functions process two encrypted integers.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.math.number_theory import (
    make_gcd, make_lcm, make_is_coprime, make_is_divisible
)

gcd_fn = make_gcd(min_value=0, max_value=20)
lcm_fn = make_lcm(min_value=0, max_value=20)
coprime_fn = make_is_coprime(min_value=0, max_value=20)
divisible_fn = make_is_divisible(0, 20, 0, 20, zero_result=999)

def test_factors(a: int, b: int):
    return gcd_fn(a, b), lcm_fn(a, b), coprime_fn(a, b), divisible_fn(a, b)

assert test_factors(12, 8) == (4, 24, 0, 0) # GCD is 4, LCM is 24, not coprime, not divisible
assert test_factors(14, 7) == (7, 14, 0, 1) # Divisible!
assert test_factors(9, 4) == (1, 36, 1, 0)  # Coprime!
assert test_factors(10, 0) == (10, 0, 0, 999) # Divisible by zero handled!
print("Cleartext factors passed!")

compiler = fhe.Compiler(test_factors, {"a": "encrypted", "b": "encrypted"})
inputset = [(0, 1), (12, 8), (14, 7), (9, 4), (10, 0)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(12, 8)
assert enc_res == (4, 24, 0, 0)
print("✅ Encrypted factors passed!")

## 2. Primes and Totients
Functions: `make_is_prime`, `make_next_prime`, `make_totient` (Euler's Totient).

In [ ]:
from concrete_fhe_toolkit.math.number_theory import (
    make_is_prime, make_next_prime, make_totient
)

prime_fn = make_is_prime(min_value=0, max_value=50)
next_prime_fn = make_next_prime(min_value=0, max_value=50)
totient_fn = make_totient(min_value=0, max_value=50)

def test_primes(n: int):
    return prime_fn(n), next_prime_fn(n), totient_fn(n)

assert test_primes(7) == (1, 11, 6) # 7 is prime, next is 11, totient is 6
assert test_primes(10) == (0, 11, 4) # 10 is not prime, next is 11, totient is 4
print("Cleartext primes passed!")

compiler = fhe.Compiler(test_primes, {"n": "encrypted"})
inputset = [(0,), (7,), (10,)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(7)
assert enc_res == (1, 11, 6)
print("✅ Encrypted primes passed!")

## 3. Roots, Distances, and Geometry
Functions: `make_isqrt` (integer square root), `make_hypot`, `make_dist` (N-dimensional euclidean distance).

In [ ]:
from concrete_fhe_toolkit.math.number_theory import (
    make_isqrt, make_hypot, make_dist
)

isqrt_fn = make_isqrt(max_value=100)
hypot_fn = make_hypot(min_value=0, max_value=10)
dist_fn = make_dist(size=2, min_value=0, max_value=10)

def test_geometry(n: int, x: int, y: int):
    # Note: dist_fn takes lists (points)
    return isqrt_fn(n), hypot_fn(x, y), dist_fn([0, 0], [x, y])

assert test_geometry(25, 3, 4) == (5, 5, 5) # sqrt(25) = 5. hypot(3,4) = 5. dist([0,0],[3,4]) = 5
print("Cleartext geometry passed!")

compiler = fhe.Compiler(test_geometry, {"n": "encrypted", "x": "encrypted", "y": "encrypted"})
inputset = [(0, 0, 0), (25, 3, 4)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(25, 3, 4)
assert enc_res == (5, 5, 5)
print("✅ Encrypted geometry passed!")